# RNN (Recurrent Neural Network)

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [2]:
sentences = [
    "I love this product",
 "This movie made me smile",
 "Service was friendly and quick",
 "Today felt bright and happy",
 "This is the best day",
 "Absolutely fantastic experience",
 "I enjoyed every single moment",
 "Great job, well done",
 "The food tasted delicious",
 "Totally recommend to everyone",
 "Very satisfied with results",
 "This worked better than expected",
 "Amazing quality and value",
 "Such a pleasant surprise",
 "I feel positive about this",
 "I hate this product",
 "This movie bored me",
 "Service was rude and slow",
 "Today was cold and lonely",
 "This is the worst day",
 "Terrible experience overall",
 "I regret buying this",
 "Very disappointed with results",
 "The food tasted awful",
 "Do not recommend this",
 "It broke after one use",
 "Not worth the money",
 "Utterly frustrating and annoying",
 "I feel negative about this",
 "Such a waste of time",
    
]

labels = [1]*15 + [0]*15
labels = np.array(labels)

In [3]:
#Maximum number of words to keep in the vocabulary
vocab_size = 2000
#Create a tokenizer
#num_words = 2000 -> keep the most frequent 2000 words
#oov_token="<OOV>"->represent unknown words with <OOV>

tok = Tokenizer(
    num_words = vocab_size,
    oov_token = "<OOV>"
)

#learn the vocabulary from the sentences
tok.fit_on_texts(sentences)

# Convert each sentence into a sequence of integer numbers
seqs = tok.texts_to_sequences(sentences)

# Find the length of the longest sequence
maxlen = max(len(s) for s in seqs)

# Make all sequences the same length by adding 0s at the end
X = pad_sequences(
    seqs,
    maxlen = maxlen,
    padding = 'post'
)

# Target/output labels
y = labels

In [4]:
X[0]

array([ 3, 26,  2,  7,  0], dtype=int32)

In [6]:
X[29]

array([21, 22, 85, 86, 87], dtype=int32)

In [8]:
#Embedding converts word IDs into dense, learnable vectors so that the neural network can work with meaningful representation of words
#Dimension of the word embedding vector
embed_dim = 16

#Number of units (neurons) in the RNN layer
rnn_units = 8

array([ 3, 26,  2,  7,  0], dtype=int32)

In [11]:
#Create RNN Model

# Input layer
# Each sentence is represented as a sequence of integers
# maxlen = maximum length of the sequence
inp = Input(
    shape = (maxlen,),
    dtype = "int32",
    name = "input",
)

# Embedding layer
# Converts each word ID into a dense vector of size embed_dim
# mask_zero=True → ignores padding zeros

x = Embedding(
    input_dim = vocab_size,
    output_dim = embed_dim,
    mask_zero = True,
    name = "embed"
)(inp)

# Simple RNN layer
# units = number of RNN neurons
# return_sequences=False → return only the final output
# return_state=False → do not separately return the final hidden state

rnn = SimpleRNN(
    units = rnn_units,
    return_sequences = False,
    return_state = False,
    name = "simple_rnn"
)

x_last = rnn(x)

# Output layer
# Dense(1) → produces one output value
# sigmoid → converts the output into a probability between 0 and 1

out = Dense(
    1,
    activation = "sigmoid",
    name = "out"
)(x_last)

#Create the complete model
model = Model(
    inputs = inp,
    outputs = out
)

#Configure the model for training
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

#Display the model architecture
model.summary()






Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)            │ (None, 5)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embed (Embedding)             │ (None, 5, 16)             │          32,000 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal_1 (NotEqual)        │ (None, 5)                 │               0 │ input[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ simple_rnn (SimpleRNN)        │ (None, 8)                 │             200 │ embed[0][0],               │
│                               │                           │                 │ not_equal_1[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ out (Dense)                   │ (None, 1)                 │               9 │ simple_rnn[0][0]           │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
#Train the RNN Model
history = model.fit(
    X,
    y,
    epochs = 25,
    batch_size = 8,
    verbose = 1
)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.4333 - loss: 0.7091
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5000 - loss: 0.6912
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5667 - loss: 0.6758
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6667 - loss: 0.6617
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.7000 - loss: 0.6467
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7667 - loss: 0.6311
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8000 - loss: 0.6149
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9333 - loss: 0.5982
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9667 - loss: 0.5796
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9667 - loss: 0.5614
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9667 - loss: 0.5408
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9667 - loss: 0.5191
E

# Create Intermediate Model

In [13]:
# Create an intermediate model
# It uses the same input as the original model
# but returns the outputs of the Embedding and SimpleRNN layers

intermediate_model = Model(
    inputs = model.inputs,
    outputs = [
        model.get_layer('embed').output, # Embedding output
        model.get_layer('simple_rnn').output # RNN hidden/output state
    ]
)

In [14]:
# GET EMBEDDING AND RNN OUTPUTS
embedding_output, rnn_output = intermediate_model.predict(X)

print("\nEmbedding output shape: ")
print(embedding_output.shape)

print("\nRNN output shape: ")
print(rnn_output.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 450ms/step

Embedding output shape: 
(30, 5, 16)

RNN output shape: 
(30, 8)


In [18]:
# ============================================================
# CREATE RNN FOR HIDDEN-STATE INSPECTION
# ============================================================

# Create a new input layer
seq_inp = Input(
    shape=(maxlen,),
    dtype='int32'
)

# Reuse the trained Embedding layer
seq_emb = model.get_layer('embed')(seq_inp)

# Create a new SimpleRNN
# return_sequences=True → return hidden state at EVERY timestep
rnn_seq = SimpleRNN(
    units=rnn_units,
    return_sequences=True,
    name='rnn_seq'
)
# Pass embedding output through the new RNN
# The RNN is automatically built here
seq_hidden = rnn_seq(seq_emb)

In [19]:
# ============================================================
#  COPY TRAINED RNN WEIGHTS
# ============================================================

try:

    # Get weights from the trained original RNN
    trained_weights = model.get_layer(
        'simple_rnn'
    ).get_weights()

    # Copy those weights into the new RNN
    rnn_seq.set_weights(trained_weights)

    print("\nCopied RNN weights successfully.")

except Exception as e:

    print(
        "\nCould not copy weights automatically:",
        e
    )


Copied RNN weights successfully.


In [20]:
# ============================================================
#  CREATE INSPECTION MODEL
# ============================================================

# This model returns the hidden state
# at every timestep
inspect_model = Model(
    inputs=seq_inp,
    outputs=seq_hidden
)


# ============================================================
# 10. INSPECT ONE SENTENCE
# ============================================================

# Select the first sentence
idx = 0

# Keep batch dimension
# Shape: (1, maxlen)
example_seq = X[idx:idx + 1]

# Predict hidden states
hidden_seq = inspect_model.predict(
    example_seq
)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step
